## Step 0. Dataset Construction: Clinical-Imaging Linkage

This step builds the final labelled dataset by linking the imaging file
(`cxr_discharge_info.csv`) with three supplementary clinical files
(`COVID_DSL_01.CSV`, `CDSL_01.csv`, `01.csv`).

**Why this step is necessary:** the imaging file contains patient ID,
filename, admission date, and outcome (death/home), but does **not**
include the discharge date. Without the discharge date it is not possible
to determine whether a death occurred within 30 days of admission, which
is the target label for this study. The three clinical files contain
admission and discharge dates but no image filenames, so a record linkage
step is required.

**Date parsing:** the three clinical files use different date formats
(ISO `YYYY-MM-DD` in `COVID_DSL_01` and `CDSL_01`; Spanish `DD/MM/YYYY`
in `01.csv`). A two-pass explicit parser is used to handle all formats
unambiguously, avoiding the `dayfirst=True` heuristic which can
misinterpret ambiguous dates depending on the environment.

**Record linkage strategy:** for each patient, the clinical record whose
admission date is closest to the imaging admission date is selected.
Among ties, `COVID_DSL_01` (the most recent release, 2021) is preferred.
Matches with a date difference > 7 days are excluded as they cannot be
reliably attributed to the same admission episode.

**Outcome:** 1,112 patients with reliable clinical linkage (3 excluded
due to ambiguous date match), 98 with 30-day in-hospital mortality
(class 1) and 1,014 survivors (class 0); approximate 10:1 class imbalance.

In [2]:
import pandas as pd
import numpy as np
import os
 
# ===============================
# 1. Load clinical files
# ===============================
 
covid_path = "Segmented_images/discharge_info_cdsl/19_04_2021/COVID_DSL_01.CSV"
cdsl_path  = "Segmented_images/discharge_info_cdsl/20_07_2020/CDSL_01.csv"
csv01_path = "Segmented_images/discharge_info_cdsl/24_04_2020/01.csv"
 
df_covid = pd.read_csv(covid_path, sep='|')
df_cdsl  = pd.read_csv(cdsl_path, sep=';')
df_01    = pd.read_csv(csv01_path, encoding="latin1", sep=';')
 
for df in [df_covid, df_cdsl, df_01]:
    df.columns = df.columns.str.strip()
 
# ===============================
# 2. Homogenise columns
# ===============================
 
df_covid = df_covid.rename(columns={
    "IDCDSL":          "PATIENT_ID",
    "F_INGRESO_ING":   "FECHA_INGRESO",
    "F_ALTA_ING":      "FECHA_ALTA",
    "MOTIVO_ALTA_ING": "MOTIVO_ALTA"
})
df_cdsl = df_cdsl.rename(columns={
    "PATIENT ID":                        "PATIENT_ID",
    "F_INGRESO/ADMISSION_D_ING/INPAT":  "FECHA_INGRESO",
    "F_ALTA/DISCHARGE_DATE_ING":         "FECHA_ALTA",
    "MOTIVO_ALTA/DESTINY_DISCHARGE_ING": "MOTIVO_ALTA"
})
df_01 = df_01.rename(columns={
    "PATIENT ID":                        "PATIENT_ID",
    "F_INGRESO/ADMISSION_D_ING/INPAT":  "FECHA_INGRESO",
    "F_ALTA/DISCHARGE_DATE_ING":         "FECHA_ALTA",
    "MOTIVO_ALTA/DESTINY_DISCHARGE_ING": "MOTIVO_ALTA"
})
 
# ===============================
# 3. Parse dates and compute length of stay
#
# The three clinical files use different date formats:
#   - COVID_DSL_01 : "2020-04-05 00:00:00 " (YYYY-MM-DD, trailing space)
#   - CDSL_01      : "2020-04-06"            (YYYY-MM-DD)
#   - 01.csv       : "26/12/2019"            (DD/MM/YYYY)
#
# A two-pass parser handles all three formats unambiguously.
# ===============================
 
def parse_fecha(series):
    """
    Robust date parser for mixed-format clinical date columns.
    First pass: ISO format (YYYY-MM-DD) for COVID_DSL_01 and CDSL_01.
    Second pass: Spanish format (DD/MM/YYYY) for 01.csv.
    Strips whitespace to handle trailing spaces.
    """
    series = series.astype(str).str.strip()
    result = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
    mask   = result.isna()
    result[mask] = pd.to_datetime(
        series[mask], format="%d/%m/%Y", errors="coerce"
    )
    return result
 
for df in [df_covid, df_cdsl, df_01]:
    df["FECHA_INGRESO"] = parse_fecha(df["FECHA_INGRESO"])
    df["FECHA_ALTA"]    = parse_fecha(df["FECHA_ALTA"])
    df["dias_estancia"] = (df["FECHA_ALTA"] - df["FECHA_INGRESO"]).dt.days
 
# ===============================
# 4. Merge clinical files with source priority
#
# The three files cover overlapping patient cohorts and the same patient
# may appear with slightly different admission dates across files.
# COVID_DSL_01 (released 2021) is the most complete and up-to-date source
# and its admission dates best match those in the imaging file (cxr).
# Priority: COVID_DSL_01 (0) > CDSL_01 (1) > 01.csv (2).
# ===============================
 
df_covid["_prio"] = 0
df_cdsl["_prio"]  = 1
df_01["_prio"]    = 2
 
cols_keep = ["PATIENT_ID", "FECHA_INGRESO", "FECHA_ALTA",
             "dias_estancia", "MOTIVO_ALTA", "_prio"]
 
df_clinico = pd.concat(
    [df_covid[cols_keep], df_cdsl[cols_keep], df_01[cols_keep]],
    ignore_index=True
)
 
# Remove rows with missing admission or discharge date
df_clinico = df_clinico.dropna(subset=["FECHA_INGRESO", "FECHA_ALTA"])
 
# ===============================
# 5. Load imaging file
#
# cxr_discharge_info.csv contains: PatientID, Filename, StudyDate,
# AdmissionDate, DaysDifference (StudyDate - AdmissionDate),
# DischargeReason (home/death), Outcome (0/1).
# It does NOT contain discharge date, so we need the clinical files
# to compute length of stay and derive the 30-day mortality label.
# ===============================
 
df_cxr = pd.read_csv(
    "Segmented_images/discharge_info_cdsl/cxr_discharge_info.csv"
)
df_cxr.columns = df_cxr.columns.str.strip()
 
df_cxr["AdmissionDate"] = pd.to_datetime(
    df_cxr["AdmissionDate"], format="%Y-%m-%d", errors="coerce"
)
 
df_cxr = df_cxr.rename(columns={
    "PatientID":     "PATIENT_ID",
    "AdmissionDate": "FECHA_INGRESO"
})
 
# ===============================
# 6. Merge imaging and clinical data
#
# The admission date in cxr may differ by a few days from the clinical
# files because the X-ray is taken after admission (median DaysDifference=2)
# and dates may be recorded slightly differently across hospital systems.
#
# Strategy:
#   1. For each patient, find the clinical record whose admission date is
#      closest to the imaging admission date.
#   2. Among ties, prefer the highest-priority source (COVID_DSL_01).
#   3. Accept matches within 7 days (clinically justified: the X-ray is
#      taken within the first week of admission in all cases,
#      DaysDifference max = 11 days in the imaging file itself).
#   4. The 3 patients whose best match exceeds 7 days are excluded,
#      as the match cannot be reliably attributed to the same admission.
#
# This yields 1,112 patients with reliable clinical linkage.
# Note: the original Anaconda script using dayfirst=True produced 1,115
# patients and 98 deaths, but that parser misread some ambiguous dates,
# producing spurious exact matches. The present approach is more conservative
# and methodologically transparent.
# ===============================
 
merged = df_cxr[[
    "PATIENT_ID", "FECHA_INGRESO", "Filename", "StudyDate",
    "DaysDifference", "DischargeReason", "Outcome"
]].merge(
    df_clinico[[
        "PATIENT_ID", "FECHA_INGRESO", "FECHA_ALTA",
        "dias_estancia", "MOTIVO_ALTA", "_prio"
    ]],
    on="PATIENT_ID",
    how="left",
    suffixes=("_cxr", "_cli")
)
 
# Compute date difference between imaging and clinical admission dates
merged["diff_dias"] = abs(
    (merged["FECHA_INGRESO_cxr"] - merged["FECHA_INGRESO_cli"]).dt.days
)
 
# Drop rows where diff_dias could not be computed
merged = merged.dropna(subset=["diff_dias"])
 
# For each patient, keep the closest clinical record (prioritising source)
merged = merged.sort_values(["PATIENT_ID", "diff_dias", "_prio"])
df_final = merged.drop_duplicates(subset=["PATIENT_ID"], keep="first").copy()
 
# Exclude patients whose best clinical match is > 7 days away
n_before = len(df_final)
df_final = df_final[df_final["diff_dias"] <= 7].reset_index(drop=True)
n_excluded = n_before - len(df_final)
print(f"Patients excluded (best match > 7 days): {n_excluded}")
 
df_final = df_final.rename(columns={"FECHA_INGRESO_cxr": "FECHA_INGRESO"})
df_final = df_final.drop(columns=["FECHA_INGRESO_cli", "_prio"])
 
# ===============================
# 7. Create 30-day mortality label
#
# A patient is labelled deceased (mortalidad_30d = 1) if:
#   - Outcome == 1 (in-hospital death recorded in imaging file)
#   - Length of stay <= 30 days (derived from clinical discharge date)
# All other patients are labelled as survivors (mortalidad_30d = 0).
# ===============================
 
df_final["mortalidad_30d"] = 0
 
df_final.loc[
    (df_final["Outcome"] == 1) &
    (df_final["dias_estancia"] <= 30),
    "mortalidad_30d"
] = 1
 
# ===============================
# 8. Dataset diagnostics
# ===============================
 
print("\n===== DATASET DIAGNOSTICS =====")
print("Total images:                  ", len(df_final))
print("Unique patients:               ", df_final["PATIENT_ID"].nunique())
print("Unique filenames:              ", df_final["Filename"].nunique())
print("Outcome = 1 (any death):       ", (df_final["Outcome"] == 1).sum())
print("30-day mortality (n):          ", df_final["mortalidad_30d"].sum())
print("30-day mortality (%):          ",
      round(df_final["mortalidad_30d"].mean() * 100, 1), "%")
print("Images without clinical match: ",
      df_final["dias_estancia"].isna().sum())
print("Patients with diff_dias > 0:   ",
      (df_final["diff_dias"] > 0).sum())
print("Max diff_dias accepted:        ",
      df_final["diff_dias"].max())
 
images_per_patient = df_final.groupby("PATIENT_ID")["Filename"].count()
print("Patients with >1 image:        ", (images_per_patient > 1).sum())
 
print("\nLength of stay among deceased (mortalidad_30d=1):")
print(df_final[df_final["mortalidad_30d"] == 1]["dias_estancia"].describe())
 
# ===============================
# 9. Save final dataset
# ===============================
 
cols_output = [
    "PATIENT_ID", "Filename", "StudyDate", "FECHA_INGRESO",
    "FECHA_ALTA", "dias_estancia", "DaysDifference",
    "DischargeReason", "MOTIVO_ALTA", "Outcome",
    "mortalidad_30d", "diff_dias"
]
 
df_final[cols_output].to_csv("dataset_mortalidad.csv", index=False)

Patients excluded (best match > 7 days): 3

===== DATASET DIAGNOSTICS =====
Total images:                   1112
Unique patients:                1112
Unique filenames:               1112
Outcome = 1 (any death):        109
30-day mortality (n):           98
30-day mortality (%):           8.8 %
Images without clinical match:  0
Patients with diff_dias > 0:    65
Max diff_dias accepted:         7
Patients with >1 image:         0

Length of stay among deceased (mortalidad_30d=1):
count    98.000000
mean     11.867347
std       6.719164
min       1.000000
25%       7.000000
50%      10.500000
75%      16.000000
max      28.000000
Name: dias_estancia, dtype: float64
